# Dataset Selection and Preprocessing for Offline RL
This notebook selects representative episodes from the Adroit tasks (`relocate`, `door`, `hammer`, and `pen`) 
using clustering on spatial features. The selected episodes are saved in a format compatible with `d3rlpy`.

## Imports
We begin by importing all the required libraries and verifying library versions.

In [ ]:
import minari
import d3rlpy
import numpy as np
from sklearn.cluster import KMeans
import os

## Task Configuration
We define the task names and initialize empty containers to store the datasets and selected episodes.

In [ ]:
tasks = ['relocate', 'door', 'hammer', 'pen']

# Initialize dictionaries to store datasets and episodes
datasets = {}
dataset_episodes = {}

# Number of episodes to extract from each dataset
n_episodes = 50

dataset_type = 'expert' # expert - cloned - human

## Episode Selection Function
This function selects the best episode (in terms of total reward) from each spatial cluster. 
Clustering is done using the initial positional features that are task-specific.

In [ ]:
def select_balanced_episodes(task, minari_dataset, n_clusters=n_episodes):
    reward_index_pos = []

    # Iterate through all episodes
    for i, episode in enumerate(minari_dataset.iterate_episodes()):
        tot_reward = np.sum(episode.rewards)

        # Extract task-specific spatial features for clustering
        if task == 'relocate':
            pos_diff = episode.observations[0][36:39]  # ball-to-target
        elif task == 'door':
            pos_diff = episode.observations[0][35:38]  # palm-to-door-handle
        elif task == 'hammer':
            hammer_pos = episode.observations[0][36:39]
            nail_pos = episode.observations[0][42:45]
            pos_diff = hammer_pos - nail_pos           # hammer-to-nail
        elif task == "pen":
            pos_diff = episode.observations[0][39:45]  # pen-to-target (linear + angular)

        reward_index_pos.append((tot_reward, i, pos_diff))

    # Cluster based on spatial features
    positions = np.array([x[2] for x in reward_index_pos])
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(positions)
    cluster_labels = kmeans.labels_

    # Select the best episode (highest reward) per cluster
    cluster_episodes = {}
    for label, (reward, idx, _) in zip(cluster_labels, reward_index_pos):
        if label not in cluster_episodes or reward > cluster_episodes[label][0]:
            cluster_episodes[label] = (reward, idx)

    selected_indices = [v[1] for v in cluster_episodes.values()]
    print(f"Task {task} — selected {len(selected_indices)} episodes (best per cluster from full dataset)")
    return selected_indices

## Dataset Preparation Function
This function builds a `d3rlpy`-compatible dataset by extracting transitions from the selected episodes. 
It optionally renders the episodes for qualitative inspection, then saves the result as a `.npz` file.

In [ ]:
def prepare_d3_dataset(task, selected_indices, minari_dataset, visualize):
    observations = []
    actions = []
    rewards = []
    terminals = []

    # Create the environment for optional visualization
    env = minari_dataset.recover_environment(render_mode="human", camera_id=4)

    for i, episode in enumerate(minari_dataset.iterate_episodes()):
        if i not in selected_indices:
            continue  # Skip episodes not selected

        if visualize:
            # Optional rollout to visualize episode execution
            obs, _ = env.reset()
            tot_reward = 0
            for step_idx, action in enumerate(episode.actions):
                obs, reward, terminated, truncated, _ = env.step(action)
                tot_reward += reward
                if terminated or truncated:
                    print(f"\nEpisode {i} — Total reward: {tot_reward:.2f}")
                    break

        # Extract episode data
        obs_data = episode.observations[:-1]
        actions_data = episode.actions
        rewards_data = episode.rewards
        dones = np.array(episode.terminations) | np.array(episode.truncations)

        # Append to global dataset
        observations.append(obs_data)
        actions.append(actions_data)
        rewards.append(rewards_data)
        terminals.append(dones)

    env.close()

    # Concatenate all episode data into single arrays
    observations = np.concatenate(observations)
    actions = np.concatenate(actions)
    rewards = np.concatenate(rewards)
    terminals = np.concatenate(terminals)

    print(f"Task {task}  dataset built")

    # Save the dataset to a compressed .npz file
    save_path = f"datasets/{task}.npz"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    np.savez_compressed(save_path,
                        observations=observations,
                        actions=actions,
                        rewards=rewards,
                        terminals=terminals)

    print(f"Task {task} dataset saved to: {save_path}")


## Execution: Build Datasets for All Tasks
We loop over all tasks, select balanced episodes, convert them into a flat format, and save them.

In [ ]:
for task in tasks:
    # Load the specified Minari dataset for the current task
    datasets[task] = minari.load_dataset(f"D4RL/{task}/{dataset_type}-v2")
    
    # Select a balanced subset of episodes using clustering
    dataset_episodes[task] = select_balanced_episodes(task, datasets[task])
    
    # Build and save the dataset in .npz format (without visualization)
    prepare_d3_dataset(task, dataset_episodes[task], datasets[task], False)
